# training_v2_segmented_temporal
Shared repository workflow for two segment-specific models with nested past-only calibration. The untouched test remains locked.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/Qauntify/qauntify_webV1.git'
REPO_COMMIT = 'bbbd562'
REPO_DIR = '/content/qauntify_webV1'
DRIVE_ROOT = '/content/drive/MyDrive/Quantify/training_v1_full_001'
DATASET_ROOT = f'{DRIVE_ROOT}/datasets/datasets/training_v1'
EXPERIMENT_DIR = f'{DRIVE_ROOT}/training/training_v2_segmented_temporal'
RUN_SMOKE = True
RUN_FULL = False

In [ ]:
import pathlib, subprocess, sys
if not pathlib.Path(REPO_DIR, '.git').is_dir(): subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', '--detach', REPO_COMMIT], check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements-training.txt'], check=True)

In [ ]:
assert pathlib.Path(DATASET_ROOT, 'training_manifest.json').is_file(), DATASET_ROOT
assert not pathlib.Path(EXPERIMENT_DIR, 'test_evaluation_state.json').exists()
subprocess.run([sys.executable, '-m', 'pytest', 'tests/ml/segmented_training/test_segmented_temporal.py', '-q', '--confcutdir', 'tests/ml/segmented_training'], cwd=REPO_DIR, check=True)

In [ ]:
if RUN_SMOKE:
    smoke_dir = f'{EXPERIMENT_DIR}_smoke'
    command = [sys.executable, '-m', 'ml.segmented_training.cli', '--config', 'ml/configs/training_v2_segmented_temporal.yaml', '--dataset-root', DATASET_ROOT, '--experiment-dir', smoke_dir, '--smoke']
    if pathlib.Path(smoke_dir, 'run_state.json').exists(): command.append('--resume')
    subprocess.run(command, cwd=REPO_DIR, check=True)

In [ ]:
assert RUN_FULL, 'Set RUN_FULL=True only after smoke artifacts are reviewed.'
command = [sys.executable, '-m', 'ml.segmented_training.cli', '--config', 'ml/configs/training_v2_segmented_temporal.yaml', '--dataset-root', DATASET_ROOT, '--experiment-dir', EXPERIMENT_DIR]
if pathlib.Path(EXPERIMENT_DIR, 'run_state.json').exists(): command.append('--resume')
subprocess.run(command, cwd=REPO_DIR, check=True)

In [ ]:
report = pathlib.Path(EXPERIMENT_DIR, 'training_v2_report.md')
print(report.read_text() if report.is_file() else 'Full experiment has not been run.')